# pandas-ta-classic Reduced Feature Family Strategy EDA

This notebook tests two reduction strategies for the large pandas-ta-classic families:

1. Use one default-parameter call per indicator for `technical_momentum` and `technical_overlap`.
2. Prune redundant multi-output columns, especially band/channel middle lines and raw OHLC price transforms that duplicate other overlap features.

The goal is to reduce transformer feature-family width while preserving or improving family-level predictive power.


In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import sys
import warnings

import numpy as np
import polars as pl
from IPython.display import Markdown, display

_current = Path.cwd().resolve()
_repo_candidates = [_current, *_current.parents]
REPO_ROOT = next((path for path in _repo_candidates if (path / "quant_warehouse").exists() and (path / "pyproject.toml").exists()), _current)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas_ta_classic as ta
from quant_warehouse import Warehouse
from quant_warehouse.platforms.data_providers.fmp.feature_engineering import TA_CLASSIC_FAMILY_PREFIXES
from quant_warehouse.platforms.data_providers.fmp.feature_engineering.ta_classic_technical import (
    _auto_indicator_spec,
    _compute_indicator,
    _feature_column_name,
    _indicator_specs,
    _prepare_price_frame,
    _to_built_feature_set,
)
from quant_warehouse.ingest.credentials import configure_openbb_credentials

PROVIDER = "fmp"
ANALYSIS_LABEL = "OpenBB/FMP screened US >= $100B universe"
SCREEN_MARKET_CAP_MIN = 100_000_000_000
SCREEN_COUNTRY = "US"
SCREEN_EXCHANGES = ("NASDAQ", "NYSE", "AMEX")
SCREEN_LIMIT = 5_000
START_DATE = "2018-01-01"
END_DATE = None
HORIZONS = (20, 60, 120)
MIN_OBS = 120

wh = Warehouse()
run_timings: dict[str, float] = {}


## Strategy Definitions


In [2]:
def default_large_family_specs() -> dict[str, list]:
    specs = {"technical_momentum": [], "technical_overlap": []}
    for category, family in [("momentum", "technical_momentum"), ("overlap", "technical_overlap"), ("volatility", "technical_overlap")]:
        for fn_name in ta.Category.get(category, []):
            spec = _auto_indicator_spec(ta, fn_name)
            if spec is not None:
                specs[family].append(spec)
    return specs


def keep_reduced_output_column(family: str, spec_name: str, raw_column: str) -> bool:
    c = str(raw_column).lower()
    s = str(spec_name).lower()
    if any(x in s for x in ["bbands", "kc", "donchian", "accbands", "aberration"]):
        return any(token in c for token in ["bbp", "bbb", "width", "percent", "bandwidth"])
    if "ichimoku" in s:
        return any(token in c for token in ["isa", "isb"])
    if "supertrend" in s:
        return "trend" in c or c.startswith("supert_")
    if "psar" in s:
        return "psarl" not in c and "psars" not in c
    if any(x in s for x in ["rainbow", "mmar"]):
        return any(token in c for token in ["osc", "width", "signal"])
    if s in {"hl2", "hlc3", "ohlc4", "avgprice", "medprice", "typprice", "wcp"}:
        return False
    return True


def reduced_indicator_specs() -> dict[str, list]:
    full = {family: list(specs) for family, specs in _indicator_specs(ta).items()}
    reduced = dict(full)
    large_defaults = default_large_family_specs()
    reduced["technical_momentum"] = large_defaults["technical_momentum"]
    reduced["technical_overlap"] = large_defaults["technical_overlap"]
    return reduced


def _count_outputs_for_symbol(symbol: str = "AAPL") -> pl.DataFrame:
    prices = _prepare_price_frame(wh.read_prices(symbol, provider=PROVIDER).loc[START_DATE:])
    rows = []
    strategies = {
        "current_full_wrapper": _indicator_specs(ta),
        "default_large_families": {**_indicator_specs(ta), **default_large_family_specs()},
        "default_plus_pruned_outputs": reduced_indicator_specs(),
    }
    for strategy, specs_by_family in strategies.items():
        for family, specs in specs_by_family.items():
            columns = []
            for spec in specs:
                indicator = _compute_indicator(ta, prices, spec)
                for column in indicator.columns:
                    if strategy == "default_plus_pruned_outputs" and family in {"technical_momentum", "technical_overlap"}:
                        if not keep_reduced_output_column(family, spec.name, str(column)):
                            continue
                    columns.append(str(column))
            rows.append({"strategy": strategy, "family": family, "feature_count": len(columns), "indicator_specs": len(specs)})
    return pl.DataFrame(rows).sort_values(["strategy", "feature_count"], ascending=[True, False])

count_comparison = _count_outputs_for_symbol("AAPL")
display(count_comparison)
display(count_comparison.pivot(index="family", columns="strategy", values="feature_count").fillna(0).astype(int))


,strategy,family,feature_count,indicator_specs
3,current_full_wrapper,technical_momentum,178,98
4,current_full_wrapper,technical_overlap,112,66
0,current_full_wrapper,technical_candles,72,5
2,current_full_wrapper,technical_math,25,19
1,current_full_wrapper,technical_cycles,11,8
5,current_full_wrapper,technical_performance,9,7
10,default_large_families,technical_overlap,106,64
9,default_large_families,technical_momentum,96,52
6,default_large_families,technical_candles,72,5
8,default_large_families,technical_math,25,19


strategy,current_full_wrapper,default_large_families,default_plus_pruned_outputs
family,,,
technical_candles,72,72,72
technical_cycles,11,11,11
technical_math,25,25,25
technical_momentum,178,96,96
technical_overlap,112,106,65
technical_performance,9,9,9


## Universe


In [3]:
FALLBACK_SYMBOLS = (
    "AAPL", "ABBV", "ABT", "ADI", "AMAT", "AMD", "AMGN", "AMZN", "ANET", "APH", "APP", "AVGO", "AXP", "BA", "BAC", "BKNG", "BLK", "BMY", "BRK-A", "BRK-B", "BX", "C", "CAT", "CDNS", "COF", "COP", "COST", "CRM", "CRWD", "CSCO", "CVS", "CVX", "DE", "DELL", "DHR", "DIS", "DUK", "EQIX", "FTNT", "GE", "GEV", "GILD", "GLW", "GOOG", "GOOGL", "GS", "HD", "HONIV", "HWM", "IBKR", "IBM", "INTC", "ISRG", "JNJ", "JPM", "KLAC", "KO", "LLY", "LMT", "LOW", "LRCX", "MA", "MCD", "MDT", "META", "MO", "MRK", "MRVL", "MS", "MSFT", "MU", "NEE", "NEM", "NFLX", "NOW", "NVDA", "ORCL", "PANW", "PEP", "PFE", "PG", "PGR", "PH", "PLD", "PLTR", "PM", "PWR", "QCOM", "RTX", "SBUX", "SCCO", "SCHW", "SNDK", "SO", "SOJE", "SOMN", "SPGI", "SYK", "T", "TBB", "TJX", "TMO", "TMUS", "TSLA", "TXN", "UBER", "UNH", "UNP", "V", "VRT", "VRTX", "VZ", "WDC", "WELL", "WFC", "WMT", "XOM"
)


def _normalize_screener_frame(frame: pl.DataFrame) -> pl.DataFrame:
    out = frame.copy()
    out.columns = [str(c).strip() for c in out.columns]
    if "symbol" not in out.columns:
        for candidate in ("ticker", "Symbol", "Ticker"):
            if candidate in out.columns:
                out = out.rename(columns={candidate: "symbol"})
                break
    if "symbol" not in out.columns:
        return pl.DataFrame(columns=["symbol"])
    out["symbol"] = out["symbol"].astype(str).str.strip().str.upper()
    return out.dropna(subset=["symbol"]).drop_duplicates("symbol")


def fetch_openbb_fmp_screener(exchange: str) -> pl.DataFrame:
    from openbb import obb
    configure_openbb_credentials()
    data = obb.equity.search(
        provider="fmp",
        mktcap_min=SCREEN_MARKET_CAP_MIN,
        country=SCREEN_COUNTRY,
        exchange=exchange,
        is_etf=False,
        is_fund=False,
        is_active=True,
        all_share_classes=False,
        limit=SCREEN_LIMIT,
    )
    frame = data.to_df() if hasattr(data, "to_df") else pl.DataFrame(data)
    return _normalize_screener_frame(frame)

fetch_rows = []
frames = []
for exchange in SCREEN_EXCHANGES:
    try:
        frame = fetch_openbb_fmp_screener(exchange)
        frames.append(frame)
        fetch_rows.append({"exchange": exchange, "rows": len(frame), "status": "ok" if len(frame) else "empty"})
    except Exception as exc:
        fetch_rows.append({"exchange": exchange, "rows": 0, "status": type(exc).__name__})
raw_universe = pl.concat(frames, ignore_index=True).drop_duplicates("symbol") if frames else pl.DataFrame(columns=["symbol"])
if raw_universe.empty:
    raw_universe = pl.DataFrame({"symbol": FALLBACK_SYMBOLS})
    fetch_rows.append({"exchange": "fallback", "rows": len(raw_universe), "status": "used"})

eligibility = []
for symbol in raw_universe["symbol"].astype(str).str.upper().drop_duplicates():
    try:
        prices = wh.read_prices(symbol, provider=PROVIDER)
        ok = prices is not None and not prices.empty and all(c in prices.columns for c in ["open", "high", "low", "close", "volume"])
        reason = "ok" if ok else "missing_prices"
    except Exception as exc:
        ok = False
        reason = type(exc).__name__
    eligibility.append({"symbol": symbol, "eligible": ok, "reason": reason})
eligibility = pl.DataFrame(eligibility)
ANALYSIS_SYMBOLS = tuple(eligibility.loc[eligibility["eligible"], "symbol"].sort_values())

display(pl.DataFrame(fetch_rows))
display(eligibility["reason"].value_counts().rename_axis("reason").reset_index(name="symbols"))
display(Markdown(f"> Selected {len(ANALYSIS_SYMBOLS):,} symbols from `{ANALYSIS_LABEL}`."))


,exchange,rows,status
0,NASDAQ,0,OpenBBError
1,NYSE,0,OpenBBError
2,AMEX,0,OpenBBError
3,fallback,117,used


,reason,symbols
0,ok,117


> Selected 117 symbols from `OpenBB/FMP screened US >= $100B universe`.

## Build Reduced Feature Panel


In [4]:
def _expected_direction(feature: str, family: str) -> str:
    text = feature.lower()
    if family == "technical_performance":
        return "higher_is_better"
    if any(token in text for token in ["volatility", "stdev", "variance", "drawdown", "atr", "range", "ulcer", "width", "bbb"]):
        return "lower_is_better"
    if any(token in text for token in ["rsi", "stoch", "willr", "cci", "zscore", "bbp", "percent"]):
        return "lower_is_better"
    return "higher_is_better"


def _slice_prices(frame: pl.DataFrame) -> pl.DataFrame:
    if frame is None or frame.empty:
        return pl.DataFrame()
    out = frame.copy()
    out.index = pl.Series.str.to_datetime(out.index, errors="coerce")
    out = out.loc[out.index.notna()].sort_index()
    out = out.loc[out.index >= pl.Timestamp(START_DATE)]
    if END_DATE is not None:
        out = out.loc[out.index <= pl.Timestamp(END_DATE)]
    return out


def build_reduced_family_sets(symbol: str, prices: pl.DataFrame) -> dict[str, pl.DataFrame]:
    clean = _prepare_price_frame(prices)
    specs_by_family = reduced_indicator_specs()
    result = {}
    for family, specs in specs_by_family.items():
        columns = {}
        for spec in specs:
            indicator = _compute_indicator(ta, clean, spec)
            if indicator.empty:
                continue
            for raw_col in indicator.columns:
                if family in {"technical_momentum", "technical_overlap"} and not keep_reduced_output_column(family, spec.name, str(raw_col)):
                    continue
                out_col = _feature_column_name(family, spec.name, str(raw_col))
                columns[out_col] = pl.Series.cast(indicator[raw_col], errors="coerce")
        frame = pl.DataFrame(columns, index=clean.index) if columns else pl.DataFrame(index=clean.index)
        feature_cols = []
        float32_max = np.finfo(np.float32).max
        for column in frame.columns:
            series = frame[column].replace([np.inf, -np.inf], np.nan)
            if series.notna().any():
                frame[column] = pl.Series.cast(series.ffill().fillna(0.0), errors="coerce").clip(-float32_max, float32_max).fillna(0.0).astype(np.float32)
                feature_cols.append(column)
        result[family] = frame.loc[:, feature_cols] if feature_cols else pl.DataFrame(index=clean.index)
    return result


def build_symbol_panel(symbol: str):
    start = perf_counter()
    prices = _slice_prices(wh.read_prices(symbol, provider=PROVIDER))
    if prices.empty or "close" not in prices.columns:
        return pl.DataFrame(), [], {"symbol": symbol, "status": "missing_prices"}
    family_sets = build_reduced_family_sets(symbol, prices)
    panel = pl.DataFrame({"date": pl.DatetimeIndex(prices.index), "symbol": symbol, "close": pl.Series.cast(prices["close"], errors="coerce")}, index=prices.index)
    specs = []
    for family, frame in family_sets.items():
        if frame.empty:
            continue
        frame = frame.reindex(prices.index)
        for col in frame.columns:
            panel[col] = frame[col]
            specs.append({"feature": col, "family": family, "expected_direction": _expected_direction(col, family)})
    for horizon in HORIZONS:
        panel[f"forward_return_{horizon}d"] = panel["close"].shift(-horizon) / panel["close"] - 1.0
    return panel.reset_index(drop=True), specs, {"symbol": symbol, "status": "ok", "price_rows": len(prices), "feature_count": len(specs), "seconds": perf_counter() - start}

panel_start = perf_counter()
frames=[]; all_specs=[]; diagnostics=[]
for symbol in ANALYSIS_SYMBOLS:
    frame, specs, diag = build_symbol_panel(symbol)
    diagnostics.append(diag)
    if not frame.empty:
        frames.append(frame); all_specs.extend(specs)
panel = pl.concat(frames, ignore_index=True).sort_values(["date", "symbol"]).reset_index(drop=True)
feature_metadata = pl.DataFrame(all_specs).drop_duplicates().sort_values(["family", "feature"]).reset_index(drop=True)
diagnostics_df = pl.DataFrame(diagnostics)
feature_cols = feature_metadata["feature"].tolist()
run_timings["panel_build_seconds"] = perf_counter() - panel_start
run_timings["feature_build_seconds"] = float(diagnostics_df.get("seconds", pl.Series(dtype="float64")).sum())
print({"symbols": panel['symbol'].nunique(), "rows": len(panel), "features": len(feature_cols), "panel_build_seconds": round(run_timings['panel_build_seconds'], 2), "feature_build_seconds": round(run_timings['feature_build_seconds'], 2)})
display(diagnostics_df)
display(feature_metadata.groupby("family").size().rename("feature_count").reset_index().sort_values("feature_count", ascending=False))


/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

...

/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = frame[col]
/tmp/ipykernel_1949844/279826644.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy(

{'symbols': 117, 'rows': 238717, 'features': 276, 'panel_build_seconds': 101.82, 'feature_build_seconds': 101.57}


,symbol,status,price_rows,feature_count,seconds
0,AAPL,ok,2130,276,0.9029
1,ABBV,ok,2130,276,0.8989
2,ABT,ok,2130,276,0.8951
3,ADI,ok,2130,276,0.8927
4,AMAT,ok,2130,276,0.8984
...,...,...,...,...,...
112,WDC,ok,2130,276,0.8944
113,WELL,ok,2130,276,0.8962
114,WFC,ok,2130,276,0.8912
115,WMT,ok,2130,276,0.8924


,family,feature_count
3,technical_momentum,94
0,technical_candles,72
4,technical_overlap,65
2,technical_math,25
1,technical_cycles,11
5,technical_performance,9


## Evaluation


In [5]:
def _rank_2d_nan(values: np.ndarray) -> np.ndarray:
    out = np.full(values.shape, np.nan, dtype="float32")
    for i in range(values.shape[0]):
        row = values[i]
        valid = np.isfinite(row)
        count = int(valid.sum())
        if count == 0:
            continue
        order = np.argsort(row[valid], kind="mergesort")
        ranks = np.empty(count, dtype="float32")
        ranks[order] = np.arange(1, count + 1, dtype="float32")
        out[i, np.flatnonzero(valid)] = ranks
    return out

def _mean_center_nan(values: np.ndarray, axis: int) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(values, axis=axis, keepdims=True)
    return values - mean

def _build_eval_arrays(features):
    dates = pl.DatetimeIndex(sorted(panel["date"].dropna().unique()))
    symbols = sorted(panel["symbol"].dropna().unique())
    feature_values = np.full((len(dates), len(symbols), len(features)), np.nan, dtype="float32")
    for i, feature in enumerate(features):
        feature_values[:, :, i] = panel.pivot(index="date", columns="symbol", values=feature).reindex(index=dates, columns=symbols).to_numpy(dtype="float32")
    signs = feature_metadata.set_index("feature").loc[features, "expected_direction"].map({"higher_is_better": 1.0, "lower_is_better": -1.0}).to_numpy(dtype="float32")
    feature_scores = feature_values * signs.reshape(1, 1, -1)
    returns_by_horizon = {h: panel.pivot(index="date", columns="symbol", values=f"forward_return_{h}d").reindex(index=dates, columns=symbols).to_numpy(dtype="float32") for h in HORIZONS}
    return feature_scores, returns_by_horizon

def evaluate_features(features):
    start = perf_counter()
    feature_scores, returns_by_horizon = _build_eval_arrays(features)
    days, symbols, n_features = feature_scores.shape
    flat = feature_scores.transpose(0,2,1).reshape(days*n_features, symbols)
    feature_ranks = _rank_2d_nan(flat).reshape(days, n_features, symbols).transpose(0,2,1)
    centered_features = _mean_center_nan(feature_ranks, axis=1)
    rows=[]
    for horizon, returns in returns_by_horizon.items():
        rr = _rank_2d_nan(returns)
        cr = _mean_center_nan(rr, axis=1)
        valid = np.isfinite(centered_features) & np.isfinite(cr[:, :, None])
        cf = np.where(valid, centered_features, 0.0)
        cr0 = np.where(np.isfinite(cr), cr, 0.0)
        num = np.einsum("dsf,ds->df", cf, cr0)
        den = np.sqrt(np.einsum("dsf,dsf->df", cf, cf) * np.einsum("ds,ds->d", cr0, cr0)[:, None])
        daily_ic = num / den
        daily_ic[den == 0] = np.nan
        for i, feature in enumerate(features):
            fs = feature_scores[:, :, i]
            valid_pair = np.isfinite(fs) & np.isfinite(returns)
            obs = int(valid_pair.sum())
            if obs < MIN_OBS:
                continue
            spreads=[]
            for d in range(days):
                mask = valid_pair[d]
                if int(mask.sum()) < 10:
                    continue
                scores = fs[d, mask]; rets = returns[d, mask]
                lo = np.nanquantile(scores, 0.2); hi = np.nanquantile(scores, 0.8)
                spreads.append(float(np.nanmean(rets[scores >= hi]) - np.nanmean(rets[scores <= lo])))
            ic = daily_ic[:, i]
            rows.append({"feature": feature, "horizon": horizon, "mean_daily_rank_ic": float(np.nanmean(ic)), "median_daily_rank_ic": float(np.nanmedian(ic)), "rank_ic_hit_rate": float(np.nanmean(ic > 0)), "spread_bps": float(np.nanmedian(spreads) * 10000) if spreads else np.nan, "observations": obs})
    run_timings["evaluation_seconds"] = perf_counter() - start
    return pl.DataFrame(rows)

results_df = evaluate_features(feature_cols).merge(feature_metadata, on="feature", how="left")
print({"evaluation_seconds": round(run_timings["evaluation_seconds"], 2), "feature_horizon_results": len(results_df)})
summary_by_family = results_df.groupby(["horizon", "family"]).agg(features=("feature", "nunique"), mean_rank_ic=("mean_daily_rank_ic", "mean"), median_rank_ic=("mean_daily_rank_ic", "median"), median_spread_bps=("spread_bps", "median"), positive_ic_share=("mean_daily_rank_ic", lambda s: float((s > 0).mean()))).reset_index().sort_values(["horizon", "mean_rank_ic"], ascending=[True, False])
best_by_horizon = summary_by_family.groupby("horizon").head(1).reset_index(drop=True)
stable_families = summary_by_family.groupby("family").agg(horizons=("horizon", "nunique"), avg_rank_ic=("mean_rank_ic", "mean"), min_rank_ic=("mean_rank_ic", "min"), avg_spread_bps=("median_spread_bps", "mean"), positive_horizons=("mean_rank_ic", lambda s: int((s > 0).sum())), features=("features", "max")).reset_index().sort_values(["positive_horizons", "avg_rank_ic"], ascending=[False, False])
display(summary_by_family)
display(best_by_horizon)
display(stable_families)


/tmp/ipykernel_1949844/33030029.py:48: RuntimeWarning: invalid value encountered in divide
  daily_ic = num / den


/tmp/ipykernel_1949844/33030029.py:48: RuntimeWarning: invalid value encountered in divide
  daily_ic = num / den


/tmp/ipykernel_1949844/33030029.py:48: RuntimeWarning: invalid value encountered in divide
  daily_ic = num / den


{'evaluation_seconds': 91.28, 'feature_horizon_results': 828}


,horizon,family,features,mean_rank_ic,median_rank_ic,median_spread_bps,positive_ic_share
5,20,technical_performance,9,0.0036,-0.0040,3.7163,0.3333
2,20,technical_math,25,0.0015,-0.0000,0.0000,0.4800
1,20,technical_cycles,11,-0.0014,-0.0018,0.0000,0.4545
3,20,technical_momentum,94,-0.0047,-0.0048,0.0000,0.3085
0,20,technical_candles,72,-0.0141,-0.0143,0.0000,0.0000
4,20,technical_overlap,65,-0.0209,-0.0294,-91.5346,0.1231
11,60,technical_performance,9,0.0128,-0.0021,47.6669,0.3333
8,60,technical_math,25,0.0041,0.0057,0.0000,0.5200
7,60,technical_cycles,11,0.0020,-0.0010,0.0000,0.4545
9,60,technical_momentum,94,-0.0055,-0.0042,0.0000,0.2021


,horizon,family,features,mean_rank_ic,median_rank_ic,median_spread_bps,positive_ic_share
0,20,technical_performance,9,0.0036,-0.0040,3.7163,0.3333
1,60,technical_performance,9,0.0128,-0.0021,47.6669,0.3333
2,120,technical_performance,9,0.0167,-0.0022,46.2886,0.3333


,family,horizons,avg_rank_ic,min_rank_ic,avg_spread_bps,positive_horizons,features
5,technical_performance,3,0.0111,0.0036,32.5573,3,9
2,technical_math,3,0.0033,0.0015,0.0000,3,25
1,technical_cycles,3,-0.0001,-0.0014,0.0000,1,11
3,technical_momentum,3,-0.0059,-0.0076,0.0000,0,94
0,technical_candles,3,-0.0178,-0.0233,0.0000,0,72
4,technical_overlap,3,-0.0380,-0.0551,-392.5812,0,65


## Written Analysis


In [6]:
def _fmt_seconds(value):
    return "nan" if pl.Series.is_null(value) else f"{value:,.2f}s"
family_counts = feature_metadata.groupby("family").size().rename("feature_count").reset_index().sort_values("feature_count", ascending=False)
count_pivot = count_comparison.pivot(index="family", columns="strategy", values="feature_count").fillna(0).astype(int)
best_lines = [f"- {r.horizon}d: `{r.family}` mean rank IC {r.mean_rank_ic:.4f}, median spread {r.median_spread_bps:,.1f} bps, {int(r.features)} features" for r in best_by_horizon.itertuples(index=False)]
stable_lines = [f"- `{r.family}`: avg rank IC {r.avg_rank_ic:.4f}, min rank IC {r.min_rank_ic:.4f}, positive horizons {int(r.positive_horizons)}/{int(r.horizons)}, features {int(r.features)}" for r in stable_families.itertuples(index=False)]
analysis_md = "\n".join([
    f"### {ANALYSIS_LABEL} Reduced pandas-ta-classic Strategy Analysis",
    "",
    f"The reduced strategy completed on {panel['symbol'].nunique()} symbols, {len(panel):,} symbol-days, {len(feature_cols)} features, and {len(family_counts)} families.",
    f"Panel construction took {_fmt_seconds(run_timings.get('panel_build_seconds', np.nan))}; evaluation took {_fmt_seconds(run_timings.get('evaluation_seconds', np.nan))}; total measured core runtime was {_fmt_seconds(run_timings.get('panel_build_seconds', 0.0) + run_timings.get('evaluation_seconds', 0.0))}.",
    "",
    "#### Count Impact",
    f"- `technical_momentum`: full {int(count_pivot.loc['technical_momentum', 'current_full_wrapper'])}, default-only {int(count_pivot.loc['technical_momentum', 'default_large_families'])}, default+pruned {int(count_pivot.loc['technical_momentum', 'default_plus_pruned_outputs'])} on AAPL.",
    f"- `technical_overlap`: full {int(count_pivot.loc['technical_overlap', 'current_full_wrapper'])}, default-only {int(count_pivot.loc['technical_overlap', 'default_large_families'])}, default+pruned {int(count_pivot.loc['technical_overlap', 'default_plus_pruned_outputs'])} on AAPL.",
    "",
    "#### Reduced Family Width",
    *[f"- `{r.family}`: {int(r.feature_count)} features" for r in family_counts.itertuples(index=False)],
    "",
    "#### Best Family By Horizon",
    *best_lines,
    "",
    "#### Stable Family Ranking",
    *stable_lines,
    "",
    "#### Interpretation",
    "Default parameters alone reduce momentum but barely reduce overlap because overlap is mostly many distinct indicators plus multi-output bands/channels. Redundant-output pruning is the more useful second step. If this reduced version keeps similar predictive power with fewer features, promote this approach before considering more parameter sweeps.",
])
display(Markdown(analysis_md))


### OpenBB/FMP screened US >= $100B universe Reduced pandas-ta-classic Strategy Analysis

The reduced strategy completed on 117 symbols, 238,717 symbol-days, 276 features, and 6 families.
Panel construction took 101.82s; evaluation took 91.28s; total measured core runtime was 193.10s.

#### Count Impact
- `technical_momentum`: full 178, default-only 96, default+pruned 96 on AAPL.
- `technical_overlap`: full 112, default-only 106, default+pruned 65 on AAPL.

#### Reduced Family Width
- `technical_momentum`: 94 features
- `technical_candles`: 72 features
- `technical_overlap`: 65 features
- `technical_math`: 25 features
- `technical_cycles`: 11 features
- `technical_performance`: 9 features

#### Best Family By Horizon
- 20d: `technical_performance` mean rank IC 0.0036, median spread 3.7 bps, 9 features
- 60d: `technical_performance` mean rank IC 0.0128, median spread 47.7 bps, 9 features
- 120d: `technical_performance` mean rank IC 0.0167, median spread 46.3 bps, 9 features

#### Stable Family Ranking
- `technical_performance`: avg rank IC 0.0111, min rank IC 0.0036, positive horizons 3/3, features 9
- `technical_math`: avg rank IC 0.0033, min rank IC 0.0015, positive horizons 3/3, features 25
- `technical_cycles`: avg rank IC -0.0001, min rank IC -0.0014, positive horizons 1/3, features 11
- `technical_momentum`: avg rank IC -0.0059, min rank IC -0.0076, positive horizons 0/3, features 94
- `technical_candles`: avg rank IC -0.0178, min rank IC -0.0233, positive horizons 0/3, features 72
- `technical_overlap`: avg rank IC -0.0380, min rank IC -0.0551, positive horizons 0/3, features 65

#### Interpretation
Default parameters alone reduce momentum but barely reduce overlap because overlap is mostly many distinct indicators plus multi-output bands/channels. Redundant-output pruning is the more useful second step. If this reduced version keeps similar predictive power with fewer features, promote this approach before considering more parameter sweeps.